# H1-H3: первый воспроизводимый эксперимент

**Задача:** научиться проверять ограничения между двумя контрактами и не путать
правильный отказ с ошибкой. Это development diagnostic, а не доказательство H1-H3.

Notebook работает offline с уже сохранёнными ответами модели. Он ничего не
отправляет на площадки и не запускает LLM. Для нового inference есть отдельная
команда в README. Выполняйте ячейки сверху вниз.

H1: помогает ли контекст найти трудные пары? H2: помогает ли структурированный
разбор при том же тексте и модели? H3: что добавляет проверка сверх повторного вызова?

In [1]:
from pathlib import Path
import hashlib
import json
import sys
from collections import Counter

HERE = next(p / 'experiments/week1' for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'experiments/week1/data/manifest.json').exists())
sys.path.insert(0, str(HERE))
from logic import extract_contract, possible_states
from evaluate import summarize

manifest = json.loads((HERE / 'data/manifest.json').read_text())
for name, expected in {**manifest['inputs_sha256'], **manifest['prompt_sha256']}.items():
    assert hashlib.sha256((HERE / name).read_bytes()).hexdigest() == expected, name
packets = json.loads((HERE / 'data/packets.json').read_text())
refs = {r['case_id']: r for r in json.loads((HERE / 'data/reference.json').read_text())}
print('Hashes verified. Cases:', dict(Counter(r['kind'] for r in refs.values())))

Hashes verified. Cases: {'controlled_decidable': 16, 'controlled_scope_guard': 4, 'real_unlabeled': 4}


## 1. Сначала математика одного контракта

В состоянии `10` контракт A платит 1, а B платит 0. Если `10` невозможно,
то A влечёт B. Рассмотрим A: `x > 1`, B: `x >= 1`. На границе `x=1`
получается `01`. Поэтому A влечёт B, но эквивалентности нет.

В этих искусственных примерах один контекст обозначает одно свободное
вещественное число. Для реальных рынков совпадение величины, источника и срока
ещё нужно доказать по правилам.

In [2]:
case = next(p for p in packets if refs[p['case_id']]['pattern'] == 'strict_boundary')
for side in ('A', 'B'):
    print(side, case['packet'][side]['text'], sep='\n')
a, b = (extract_contract(case['packet'][side]['text']) for side in ('A', 'B'))
worlds = possible_states(a, b)
for state in ('00', '01', '10', '11'):
    print(state, worlds.get(state, 'FORBIDDEN in the stipulated scalar world'))
assert '01' in worlds and '10' not in worlds

A
Metric: Index Alpha
Source: Bulletin Alpha
Period: 2030-01
Unit: percent
YES iff the observed value is more than 1.
Settlement: binary; no exceptions.
B
Metric: Index Alpha
Source: Bulletin Alpha
Period: 2030-01
Unit: percent
YES iff the observed value is at least 1.
Settlement: binary; no exceptions.
00 {'x_a': '-1', 'x_b': '-1'}
01 {'x_a': '1', 'x_b': '1'}
10 FORBIDDEN in the stipulated scalar world
11 {'x_a': '2', 'x_b': '2'}


## 2. Пересчитать результаты, сохранив все знаменатели

Модель уже вызвана 96 раз: 24 пары × 4 LLM-ветки. direct и structured получают
один пакет. repeat и critic начинают с одного сохранённого structured output.
Числовой checker не читает эталон и не вызывает LLM.

Таблица accuracy относится только к **16 искусственным парам**, восьми
логическим шаблонам в двух доменах. Четыре scope guards и четыре реальные пары
учитываются отдельно. Во время разработки был исправлен интерфейс свидетелей;
первый неполный запуск сохранён как format calibration, а не скрыт.

In [3]:
result = summarize(HERE / 'results/observed', HERE.parents[1] / 'tmp/week1-notebook-rescore')
print(f"{'method':24} {'exact':>8} {'coverage':>10} {'false F':>9} {'F recall':>10}")
for name, row in result['methods'].items():
    m = row['controlled']
    exact = f"{m['exact_pairs']}/{m['pairs']}"
    false = f"{m['false_forbidden']}/{m['predicted_forbidden']}"
    print(f"{name:24} {exact:>8} {m['state_coverage']:10.1%} {false:>9} {m['forbidden_recall']:10.1%}")

method                      exact   coverage   false F   F recall
always_no_constraint         2/16     100.0%       0/0       0.0%
always_unknown               0/16       0.0%       0/0       0.0%
critic                       3/16     100.0%     21/33      66.7%
critic_checked              16/16     100.0%      0/18     100.0%
direct                      10/16     100.0%      6/20      77.8%
repeat                       5/16      93.8%     16/23      38.9%
rule_only                   16/16     100.0%      0/18     100.0%
structured                   5/16      93.8%      6/11      27.8%
structured_checked          16/16     100.0%      0/18     100.0%


**Как оценивать:** меньше false forbidden полезно только вместе с coverage и
recall запретов. Всегда возвращать unknown или «нет запретов» недостаточно.
Отказ не считается точно решённой парой. При нуле предсказанных запретов
precision не определена, а не равна 100%.

In [4]:
for method in ('direct', 'structured', 'repeat', 'critic'):
    row = result['methods'][method]
    print(method, 'format errors:', row['format_errors_all_24'],
          'valid scope abstentions:', row['scope_guards']['valid_all_unknown'], '/ 4',
          'real pairs:', row['real_unlabeled'])
print('Transitions (states, not pairs):')
for contrast, counts in result['transitions'].items():
    print(contrast, counts)

direct format errors: {} valid scope abstentions: 0 / 4 real pairs: {'n': 4, 'all_unknown': 3, 'valid_all_unknown': 3, 'format_errors': 0, 'asserted_forbidden_states_unvalidated': 1}
structured format errors: {} valid scope abstentions: 0 / 4 real pairs: {'n': 4, 'all_unknown': 4, 'valid_all_unknown': 4, 'format_errors': 0, 'asserted_forbidden_states_unvalidated': 0}
repeat format errors: {'schema_error': 1} valid scope abstentions: 0 / 4 real pairs: {'n': 4, 'all_unknown': 3, 'valid_all_unknown': 2, 'format_errors': 1, 'asserted_forbidden_states_unvalidated': 2}
critic format errors: {} valid scope abstentions: 0 / 4 real pairs: {'n': 4, 'all_unknown': 4, 'valid_all_unknown': 4, 'format_errors': 0, 'asserted_forbidden_states_unvalidated': 0}
Transitions (states, not pairs):
direct__structured {'introduced_known_error': 13, 'unchanged': 44, 'repaired_known_error': 3, 'to_abstain_from_correct': 2, 'to_abstain_from_wrong': 2}
structured__structured_checked {'repaired_known_error': 18, 'u

## 3. Проверить один ответ, а не только итоговую таблицу

Следующая ячейка выбирает первый по ID структурированный ответ, который не
совпал с известными состояниями. Это иллюстративный разбор после запуска,
не отдельная подтверждающая выборка. Наличие числового свидетеля в JSON ещё
не доказывает, что он действительно даёт заявленные выплаты.

In [5]:
rows = json.loads((HERE.parents[1] / 'tmp/week1-notebook-rescore/evaluated.json').read_text())
bad = sorted([r for r in rows if r['method'] == 'structured'
              and r['kind'] == 'controlled_decidable' and r['states'] != r['reference']],
             key=lambda r: r['case_id'])
if bad:
    item = bad[0]
    raw = [json.loads(line) for line in (HERE / 'results/observed/predictions.jsonl').read_text().splitlines()]
    response = next(r for r in raw if r['case_id'] == item['case_id'] and r['branch'] == 'structured')
    packet = next(p for p in packets if p['case_id'] == item['case_id'])
    print(json.dumps(packet['packet'], ensure_ascii=False, indent=2))
    print('Reference:', item['reference'], 'Prediction:', item['states'])
    print('Raw response:', response['raw_output'])
else:
    print('No structured errors on these controlled cases. This does not establish real-market accuracy.')

{
  "scope": "CONTROLLED SCALAR; artificial contract clauses, not venue data",
  "A": {
    "text": "Metric: Index Alpha\nSource: Bulletin Alpha\nPeriod: 2030-01\nUnit: percent\nYES iff the observed value is at least 1.\nSettlement: binary; no exceptions."
  },
  "B": {
    "text": "Metric: Index Alpha\nSource: Bulletin Alpha\nPeriod: 2030-01\nUnit: percent\nYES iff the observed value is at most 1.\nSettlement: binary; no exceptions."
  }
}
Reference: {'00': 'forbidden', '01': 'possible', '10': 'possible', '11': 'possible'} Prediction: {'00': 'possible', '01': 'possible', '10': 'possible', '11': 'forbidden'}
Raw response: {
  "A": {
    "metric": "Index Alpha",
    "source": "Bulletin Alpha",
    "period": "2030-01",
    "unit": "percent",
    "op": "ge",
    "threshold": "1",
    "predicate_quote": "YES iff the observed value is at least 1."
  },
  "B": {
    "metric": "Index Alpha",
    "source": "Bulletin Alpha",
    "period": "2030-01",
    "unit": "percent",
    "op": "le",
    "t

## 4. Risk-coverage и стоимость

Уменьшение риска после отказа от трудных примеров не равно исправлению этих
примеров. Пороговые кривые ниже включают все равные confidence одной группой.
Confidence не калибрована. На этой маленькой искусственной выборке не выбираем
порог для финального исследования и не считаем статистическую значимость.

In [6]:
for method in ('direct', 'structured', 'structured_checked', 'repeat', 'critic', 'critic_checked'):
    row = result['methods'][method]
    points = [(p['confidence_threshold'], round(p['state_coverage'], 3),
               p['selective_state_error']) for p in row['risk_coverage']]
    print(method, 'threshold / coverage / assessed-state error:', points)
    cost = row['cost']
    print('  calls:', cost['llm_calls'], 'input tokens:', cost['input_tokens'],
          'output tokens:', cost['output_tokens'])

direct threshold / coverage / assessed-state error: [(1.0, 0.812, 0.15384615384615385), (0.999, 0.875, 0.17857142857142858), (0.0, 1.0, 0.15625)]
  calls: 24 input tokens: 16055 output tokens: 409
structured threshold / coverage / assessed-state error: [(0.99, 0.375, 0.3333333333333333), (0.98, 0.875, 0.2857142857142857), (0.95, 0.938, 0.3), (0.9, 0.938, 0.3)]
  calls: 24 input tokens: 36503 output tokens: 6346
structured_checked threshold / coverage / assessed-state error: [(0.99, 0.375, 0.0), (0.98, 0.875, 0.0), (0.95, 0.938, 0.0), (0.9, 1.0, 0.0)]
  calls: 24 input tokens: 36503 output tokens: 6346
repeat threshold / coverage / assessed-state error: [(0.99, 0.375, 0.4583333333333333), (0.98, 0.875, 0.4107142857142857), (0.9, 0.938, 0.4166666666666667), (0.0, 0.938, 0.4166666666666667)]
  calls: 48 input tokens: 80960 output tokens: 12912
critic threshold / coverage / assessed-state error: [(0.99, 0.438, 0.4642857142857143), (0.98, 0.938, 0.4166666666666667), (0.9, 1.0, 0.421875)]
  

## 5. Начало H1: подготовленная очередь

Для H1 здесь есть 323 Kalshi-контракта, TF-IDF и MiniLM в двух представлениях,
и 40 пар для слепой разметки. Это single-venue подготовка данных. Большее число
кандидатов или изменение ранга не означает лучшего качества поиска.

Следующий содержательный шаг: собрать полные межплощадочные пакеты, независимо
просмотреть небольшие блоки, включая пары вне top-k, и только затем считать
Recall@20 на фиксированном корпусе противоположной площадки.

In [7]:
h1 = json.loads((HERE / 'h1_queue/summary.json').read_text())
print({k: h1[k] for k in ('records', 'events', 'union_pairs', 'queue_pairs')})
queue = [json.loads(line) for line in (HERE / 'h1_queue/blind_queue.jsonl').read_text().splitlines()]
print('Blind annotation queue:', len(queue), 'pairs; labels are not supplied.')

{'records': 323, 'events': 29, 'union_pairs': 2776, 'queue_pairs': 40}
Blind annotation queue: 40 pairs; labels are not supplied.


## После notebook

1. Вручную разобрать 2-3 пары из слепой очереди, сохранив источники и unknown.
2. Согласовать два независимых разбора 8-12 калибровочных пар с полными правилами.
3. Разделить dev/test по семействам событий и шаблонам до построения пар.
4. Согласовать одну основную модель, контрасты при одинаковом coverage и бюджет.

Рабочий план: `WEEK1_PLAN_RU.md`. Точный протокол и ограничения: `PROTOCOL_RU.md`.
Результаты E01 не подтверждают H1-H3 на реальных рынках и не устанавливают
арбитраж или price discovery.